# 04 — Results summary

Reads every results CSV in `reports/` and assembles the final comparison table, the figures and the written findings.

**This is the safest notebook to demo live** — it runs in seconds because it re-runs no inference, only reads results that already exist.

In [ ]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

from tabpfn_nids import config
from tabpfn_nids.evaluation import load_results

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

files = sorted(glob.glob(str(config.REPORTS_DIR / '*.csv')))
print(f'{len(files)} result files in reports/\n')
for path in files:
    print(f'  {Path(path).name:<48} {sum(1 for _ in open(path)) - 1} row(s)')

## 1. Baseline runs — every run on record

Each row carries its own provenance: seed, hardware, device, library versions, checkpoint and git commit. That is what makes a result reproducible rather than merely reported.

In [ ]:
rows = load_results('baseline')
baseline = pd.DataFrame(rows)

score_cols = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
num_cols = score_cols + ['runtime_seconds']
baseline[num_cols] = baseline[num_cols].astype(float)

baseline[['seed', 'context_rows', 'test_rows', 'n_features',
          'n_estimators'] + num_cols].round(4)

## 2. Mean and spread across seeds

Reported as mean ± standard deviation rather than with a significance test. The Wilcoxon signed-rank test cannot reach p < 0.05 with three seeds — its smallest attainable two-sided p at n=3 is 0.25 — so a p-value here would be uninformative by construction.

In [ ]:
summary = baseline[score_cols].agg(['mean', 'std']).T
summary.columns = ['mean', 'std']
summary = summary.round(4)
summary['reported'] = [f'{m:.4f} ± {s:.4f}'
                       for m, s in zip(summary['mean'], summary['std'])]
print(summary.to_string())

f1_std = baseline['f1_score'].std()
print(f'\nNOISE FLOOR: F1 std across seeds = {f1_std:.4f} ({100 * f1_std:.2f} pp)')
print(f'Any single-seed delta below ~{100 * f1_std:.1f} pp is not evidence of an effect.')

## 3. Every experiment arm

The full picture across baseline, feature-engineering ablation and the context/stratification ablations. Arms with no results are shown as *not run* rather than quietly omitted.

In [ ]:
arms = []

arms.append({
    'arm': 'Vanilla TabPFN (single context)',
    'source': 'baseline (3 seeds)',
    'features': int(baseline['n_features'].iloc[0]),
    'context_rows': int(baseline['context_rows'].iloc[0]),
    'test_rows': int(baseline['test_rows'].iloc[0]),
    **{c: baseline[c].mean() for c in score_cols},
    'runtime_s': baseline['runtime_seconds'].mean(),
})

fa = pd.DataFrame(load_results('feature_ablation'))
if len(fa):
    fa[score_cols + ['runtime_seconds']] = fa[score_cols + ['runtime_seconds']].astype(float)
    for _, row in fa.iterrows():
        arms.append({
            'arm': ('Chunked ensemble' if row['arm'] == 'baseline'
                    else 'Chunked ensemble + engineered feats'),
            'source': 'feature_ablation (1 seed)',
            'features': int(row['n_features']),
            'context_rows': int(row['context_rows']),
            'test_rows': int(row['test_rows']),
            **{c: row[c] for c in score_cols},
            'runtime_s': row['runtime_seconds'],
        })

enhanced = load_results('enhanced')
if not enhanced:
    arms.append({'arm': 'Chunked ensemble (full-scale, all seeds)',
                 'source': 'NOT RUN — scripts/run_enhanced.py',
                 'features': np.nan, 'context_rows': np.nan, 'test_rows': np.nan,
                 **{c: np.nan for c in score_cols}, 'runtime_s': np.nan})

comparison = pd.DataFrame(arms).set_index('arm')
comparison[score_cols] = comparison[score_cols].astype(float).round(4)
comparison

### The deltas that matter

Each arm compared against the vanilla baseline, with the noise floor from section 2 applied as the decision rule.

In [ ]:
reference = comparison.iloc[0]['f1_score']

print(f'Reference: vanilla TabPFN, F1 = {reference:.4f}')
print(f'Noise floor: {f1_std:.4f} F1 ({100 * f1_std:.2f} pp)\n')

for arm, row in comparison.iloc[1:].iterrows():
    if pd.isna(row['f1_score']):
        print(f'{arm:<44} not run')
        continue
    delta = row['f1_score'] - reference
    verdict = ('significant' if abs(delta) > f1_std
               else 'within noise — not an effect')
    print(f'{arm:<44} {delta:+.4f} F1 ({100 * delta:+.2f} pp)  {verdict}')

print('\nCaveat: the ensemble arms come from a single seed on 1,000 test rows, '
      '\nwhile the baseline is 3 seeds on 5,000. They are not scale-matched, '
      '\nso these deltas are indicative rather than conclusive.')

## 4. Ablations

Full detail in notebook 03; the headline rows are repeated here so this notebook stands alone.

In [ ]:
ab_files = sorted(glob.glob(str(config.REPORTS_DIR / 'ablation_*.csv')))
if ab_files:
    ablation = pd.concat([pd.read_csv(p) for p in ab_files], ignore_index=True)
    print(ablation[['label', 'chunk_size', 'stratified', 'n_chunks',
                    'f1_score', 'roc_auc', 'runtime_seconds',
                    'max_chunk_balance_drift']].round(4).to_string(index=False))
else:
    print('No ablation CSVs in reports/.')

## 5. Provenance

What produced these numbers. A reviewer should be able to read this block and know exactly what to reconstruct.

In [ ]:
first = rows[0]
for key in ('hardware', 'device', 'tabpfn_version', 'checkpoint',
            'torch_version', 'sklearn_version', 'python_version', 'git_commit'):
    print(f'  {key:<18} {first.get(key)}')

## 6. Figures

In [ ]:
figures = sorted(glob.glob(str(config.FIGURES_DIR / '*.png')))
print(f'{len(figures)} figures in {config.FIGURES_DIR}\n')
for path in figures:
    print(Path(path).name)
    display(Image(path))

## 7. Findings

**1. TabPFN v2 works out of the box on NSL-KDD, with a characteristic error profile.**
Mean F1 of about 0.75 with ROC-AUC about 0.96 across three seeds, with no tuning, no gradient training, and a fit step that takes under a second. Precision (≈0.93) far exceeds recall (≈0.63): the model is trustworthy when it raises an alarm and misses a substantial share of attacks. That gap is the direct consequence of the 17 attack types that appear only in the test split.

**2. The high ROC-AUC / low F1 gap is a threshold problem, not a ranking problem.**
A model that ranks at 0.96 AUC but scores 0.75 F1 at the default 0.5 cut is telling you the cut is in the wrong place. Threshold selection on a validation split is the single highest-value next step, and it is recorded in `docs/FUTURE_WORK.md` rather than done here — tuning it on the test set would leak.

**3. Chunking removes a hard architectural ceiling; it has not yet been shown to raise accuracy.**
TabPFN v2 caps in-context training at 10,000 rows, so a single context can use under 8% of NSL-KDD. The chunked ensemble lifts that by construction. But the full-scale, multi-seed enhanced run has not been executed, so no accuracy claim for Enhancement 1 is supported yet. That is the largest open gap in these results.

**4. Engineered features did not help.**
In the recorded ablation the 46 engineered columns *reduced* F1 by about 2.2 pp on a single seed — a difference smaller than the 2.3 pp seed-to-seed noise floor. The defensible statement is "no measurable effect", not "an improvement" and not "a regression". Reporting this as a negative result is the correct scientific outcome.

**5. A larger per-chunk context is not worth its cost on this dataset.**
Going from 1,000 to 10,000 rows per chunk multiplied runtime by roughly 65x for no F1 gain — the best F1 in the sweep was at 5,000, and all four points sit inside the noise floor.

**6. Stratified chunking is a guarantee, not an accuracy lever — here.**
It cuts chunk class-balance drift by two orders of magnitude, exactly as designed, with no measurable F1 effect on NSL-KDD's near-balanced classes. It should be kept for the imbalanced datasets (UNSW-NB15, CIC-IDS-2018) where random chunks could plausibly starve a chunk of positives.

### Limitations to state before anyone asks

- The baseline is 3 seeds; every ensemble and ablation arm is **1 seed**. Single-seed deltas below ~2.3 pp F1 carry no weight.
- Arms are scored on different test-set sizes (5,000 for baseline, 1,000 for ablations), so cross-arm comparison is indicative only.
- All results are NSL-KDD. The UNSW-NB15 and CIC-IDS-2018 directories exist but hold no results — and those are the datasets where the chunked ensemble should show its real advantage, since they are large enough that a single 10,000-row context is a severe constraint.
- No statistical test is reported, deliberately: three seeds cannot reach p < 0.05 under Wilcoxon.